In [35]:
import os
import polars as pl
import csv
import numpy as np 
import pandas as pd
import math

# Load Yelp data from raw_data directory
raw_data_path = './Yelp_JSON_2018/'
target_path = './dataset2018/'
filter_size = 23

In [36]:
# Load yelp_academic_dataset_business.json using polars
business_df = pl.read_ndjson(os.path.join(raw_data_path, 'yelp_academic_dataset_business.json'))
display(business_df.head())

business_id,name,neighborhood,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
str,str,str,str,str,str,str,f64,f64,f64,i64,i64,struct[32],str,struct[7]
"""FYWN1wneV18bWNgQjJ2GNg""","""Dental by Design""","""""","""4855 E Warner Rd, Ste B9""","""Ahwatukee""","""AZ""","""85044""",33.33069,-111.978599,4.0,22,1,"{null,null,null,null,""True"",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,""True"",null,""True"",null,null}","""Dentists, General Dentistry, H…","{""7:30-17:0"",""7:30-17:0"",""7:30-17:0"",""7:30-17:0"",""7:30-17:0"",null,null}"
"""He-G7vWjzVUysIKrfNbPUQ""","""Stephen Szabo Salon""","""""","""3101 Washington Rd""","""McMurray""","""PA""","""15317""",40.291685,-80.1049,3.0,11,1,"{null,null,null,null,""True"",null,null,null,null,null,null,null,null,""True"",null,null,null,null,""3"",null,null,null,""{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}"",null,null,null,""{'coloring': True, 'africanamerican': False, 'curly': True, 'perms': True, 'kids': True, 'extensions': True, 'asian': True, 'straightperms': True}"",""False"",""True"",null,null,null}","""Hair Stylists, Hair Salons, Me…","{""9:0-20:0"",""9:0-20:0"",""9:0-16:0"",""9:0-20:0"",""9:0-20:0"",null,""8:0-16:0""}"
"""KQPW8lFf1y5BT2MxiSZ3QA""","""Western Motor Vehicle""","""""","""6025 N 27th Ave, Ste 1""","""Phoenix""","""AZ""","""85017""",33.524903,-112.11531,1.5,18,1,null,"""Departments of Motor Vehicles,…",null
"""8DShNS-LuFqpEWIp0HxijA""","""Sports Authority""","""""","""5000 Arizona Mills Cr, Ste 435""","""Tempe""","""AZ""","""85282""",33.383147,-111.964725,3.0,9,0,"{null,null,null,null,""True"",null,null,null,null,null,null,""True"",null,null,null,null,null,null,""2"",null,null,null,""{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}"",null,null,null,null,null,null,null,null,null}","""Sporting Goods, Shopping""","{""10:0-21:0"",""10:0-21:0"",""10:0-21:0"",""10:0-21:0"",""10:0-21:0"",""11:0-19:0"",""10:0-21:0""}"
"""PfOCPjBrlQAnz__NXj9h_w""","""Brick House Tavern + Tap""","""""","""581 Howe Ave""","""Cuyahoga Falls""","""OH""","""44221""",41.119535,-81.47569,3.5,116,1,"{""full_bar"",""True"",""average"",""casual"",""True"",""{'dj': False, 'background_music': True, 'no_music': False, 'karaoke': False, 'live': False, 'video': False, 'jukebox': False}"",""{'romantic': False, 'intimate': False, 'classy': False, 'hipster': False, 'divey': False, 'touristy': False, 'trendy': False, 'upscale': False, 'casual': True}"",""True"",""False"",""free"",""False"",""True"",""True"",""True"",""True"",""False"",""True"",""True"",""2"",""False"",""{'monday': False, 'tuesday': False, 'friday': True, 'wednesday': True, 'thursday': False, 'sunday': False, 'saturday': True}"",""{'dessert': False, 'latenight': True, 'lunch': False, 'dinner': True, 'breakfast': False, 'brunch': False}"",""{'garage': False, 'street': False, 'validated': False, 'lot': True, 'valet': False}"",""False"",""outdoor"",""False"",null,null,null,null,null,null}","""American (New), Nightlife, Bar…","{""11:0-1:0"",""11:0-1:0"",""11:0-1:0"",""11:0-1:0"",""11:0-1:0"",""11:0-0:0"",""11:0-2:0""}"


In [37]:
# Create business_id to category mapping
business_categories = business_df.select(['business_id', 'categories'])

# Save business_id to category mapping to TSV file
with open(os.path.join(target_path, 'businessid2category.tsv'), 'w', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    for row in business_categories.iter_rows():
        business_id, categories = row
        if categories:  # Only write if categories exist
            writer.writerow([business_id, categories])


In [38]:
# Load yelp_academic_dataset_review.json using polars
review_df = pl.read_ndjson(os.path.join(raw_data_path, 'yelp_academic_dataset_review.json'))
display(review_df.head())

review_id,user_id,business_id,stars,date,text,useful,funny,cool
str,str,str,i64,str,str,i64,i64,i64
"""v0i_UHJMo_hPBq9bxWvW4w""","""bv2nCi5Qv5vroFiqKGopiw""","""0W4lkclzZThpx3V65bVgig""",5,"""2016-05-28""","""Love the staff, love the meat,…",0,0,0
"""vkVSCC7xljjrAI4UGfnKEQ""","""bv2nCi5Qv5vroFiqKGopiw""","""AEx2SYEUJmTxVVB18LlCwA""",5,"""2016-05-28""","""Super simple place but amazing…",0,0,0
"""n6QzIUObkYshz4dz2QRJTw""","""bv2nCi5Qv5vroFiqKGopiw""","""VR6GpWIda3SfvPC-lg9H3w""",5,"""2016-05-28""","""Small unassuming place that ch…",0,0,0
"""MV3CcKScW05u5LVfF6ok0g""","""bv2nCi5Qv5vroFiqKGopiw""","""CKC0-MOWMqoeWf6s-szl8g""",5,"""2016-05-28""","""Lester's is located in a beaut…",0,0,0
"""IXvOzsEMYtiJI0CARmj77Q""","""bv2nCi5Qv5vroFiqKGopiw""","""ACFtxLv8pGrrxMm6EgjreA""",4,"""2016-05-28""","""Love coming here. Yes the plac…",0,0,0


In [39]:
# Create reviewerID to incremental id mapping
user_ids = review_df['user_id'].unique()
print(len(user_ids), "unique users found.")
#user_id_map = {reviewer_id: idx for idx, reviewer_id in enumerate(user_ids)}

# Create asin to incremental id mapping
business_ids = review_df['business_id'].unique()
#business_id_map = {asin: idx for idx, asin in enumerate(business_ids)}
print(len(business_ids), "unique businesses found.")

# Save mappings to files
# with open(os.path.join(target_path, 'user2idx.tsv'), 'w', newline='') as f:
#     writer = csv.writer(f, delimiter='\t')
#     for reviewer_id, idx in user_id_map.items():
#         writer.writerow([reviewer_id, idx])

# with open(os.path.join(target_path, 'business2idx.tsv'), 'w', newline='') as f:
#     writer = csv.writer(f, delimiter='\t')
#     for asin, idx in business_id_map.items():
#         writer.writerow([asin, idx])

1326101 unique users found.
174567 unique businesses found.


In [40]:
# Filter items by popularity and users by activity in one pass
filtered_reviews = (
    review_df
    .with_row_index()
    .join(
        review_df.group_by('user_id').len().filter(pl.col('len') >= filter_size),
        on='user_id'
    )
    .join(
        review_df.group_by('business_id').len().filter(pl.col('len') >= filter_size),
        on='business_id'
    )
    .drop(['index', 'len'])
)

# Get unique user IDs and business IDs after filtering
user_ids = filtered_reviews['user_id'].unique()
num_users = len(user_ids)
print(len(user_ids), "users after filtering by activity.")

business_ids = filtered_reviews['business_id'].unique()
print(len(business_ids), "unique businesses found after filtering by popularity.")


# Save the mappings of user IDs and business IDs to TSV files after the filtering
user_id_map = {reviewer_id: idx for idx, reviewer_id in enumerate(user_ids)}
# Create asin to incremental id mapping
business_id_map = {asin: idx+1 for idx, asin in enumerate(business_ids)}

# Save mappings to files
with open(os.path.join(target_path, 'user2idx.tsv'), 'w', newline='') as f:
     writer = csv.writer(f, delimiter='\t')
     for reviewer_id, idx in user_id_map.items():
         writer.writerow([reviewer_id, idx])

with open(os.path.join(target_path, 'business2idx.tsv'), 'w', newline='') as f:
     writer = csv.writer(f, delimiter='\t')
     for asin, idx in business_id_map.items():
         writer.writerow([asin, idx])


# Create splits
split_1 = int(num_users * 0.8)
split_2 = int(num_users * 0.9)

# Create train/valid/test splits using polars expressions
train_users = set(user_ids[:split_1])
valid_users = set(user_ids[split_1:split_2])
test_users = set(user_ids[split_2:])

train_data = filtered_reviews.filter(pl.col('user_id').is_in(train_users))
valid_data = filtered_reviews.filter(pl.col('user_id').is_in(valid_users))
test_data = filtered_reviews.filter(pl.col('user_id').is_in(test_users))



30482 users after filtering by activity.
43776 unique businesses found after filtering by popularity.


In [41]:

# #Create validation dataframe with user history and items to predict
# validation_df = (
#     valid_data
#     .group_by('user_id')
#     .agg([
#         pl.col('business_id').map_elements(lambda x: [business_id_map[item] for item in x], return_dtype=pl.List(pl.Int64)).alias('given_user_history'),
#         pl.col('business_id').map_elements(lambda x: [business_id_map[item] for item in x], return_dtype=pl.List(pl.Int64)).alias('predicting_items')
#     ])
#     .rename({'user_id': 'user_id'})
#     .with_columns([
#         pl.col('user_id').replace_strict(user_id_map)
#     ])
# )

# display(validation_df.head())
valid_data = valid_data.sort(['user_id', 'date'])
# This is the specific snippet you provided, now modified for the desired output.
validation_df = (
    valid_data
    .group_by('user_id') # 'user_id' column is string at this point (the group key)
    .agg(
        # First, get the full list of mapped business_ids for each user
        pl.col('business_id').map_elements(
            lambda items_list: [business_id_map.get(item) for item in items_list if business_id_map.get(item) is not None],
            return_dtype=pl.List(pl.Int64)
        ).alias('full_mapped_history')
    )
    # DataFrame now has: user_id (string), full_mapped_history (List[Int64])
    .with_columns([
        # Split 'full_mapped_history' into 80% history and 20% prediction
        pl.col('full_mapped_history').map_elements(
            lambda history: history[:math.floor(len(history) * 0.8)], # First 80%
            return_dtype=pl.List(pl.Int64)
        ).alias('given_user_history'),
        pl.col('full_mapped_history').map_elements(
            lambda history: history[math.floor(len(history) * 0.8):], # Last 20%
            return_dtype=pl.List(pl.Int64)
        ).alias('predicting_items'),

        # Map the original 'user_id' (string) to its integer representation.
        # Since no .alias() is used, this replaces the content of the 'user_id' column.
        pl.col('user_id').replace_strict(user_id_map)
    ])
    # DataFrame now has: user_id (Int64), full_mapped_history (List[Int64]),
    #                    given_user_history (List[Int64]), predicting_items (List[Int64])
    .select([
        # Select the final desired columns. This also implicitly drops 'full_mapped_history'.
        pl.col('user_id'),           # This is the mapped integer user ID
        pl.col('given_user_history'),
        pl.col('predicting_items')
    ])
)



In [42]:
print(validation_df.head())

shape: (5, 3)
┌─────────┬─────────────────────────┬─────────────────────────┐
│ user_id ┆ given_user_history      ┆ predicting_items        │
│ ---     ┆ ---                     ┆ ---                     │
│ i64     ┆ list[i64]               ┆ list[i64]               │
╞═════════╪═════════════════════════╪═════════════════════════╡
│ 27178   ┆ [1184, 21869, … 31433]  ┆ [19064, 8977, … 22126]  │
│ 24515   ┆ [13969, 25826, … 28914] ┆ [2290, 25693, … 18887]  │
│ 26371   ┆ [21279, 31515, … 13680] ┆ [24439, 32351, … 42377] │
│ 27393   ┆ [24749, 34941, … 17326] ┆ [21972, 15484, … 31539] │
│ 27254   ┆ [36373, 21766, … 26647] ┆ [16663, 11320, … 37940] │
└─────────┴─────────────────────────┴─────────────────────────┘


In [43]:
# Create validation dataframe with user history and items to predict
# test_df = (
#     test_data
#     .group_by('user_id')
#     .agg([
#         pl.col('business_id').map_elements(lambda x: [business_id_map[item] for item in x], return_dtype=pl.List(pl.Int64)).alias('given_user_history'),
#         pl.col('business_id').map_elements(lambda x: [business_id_map[item] for item in x], return_dtype=pl.List(pl.Int64)).alias('predicting_items')
#     ])
#     .rename({'user_id': 'user_id'})
#     .with_columns([
#         pl.col('user_id').replace_strict(user_id_map)
#     ])
# )

# display(validation_df.head())
# This is the specific snippet you provided, now modified for the desired output.
test_data = test_data.sort(['user_id', 'date'])
test_df = (
    test_data
    .group_by('user_id') # 'user_id' column is string at this point (the group key)
    .agg(
        # First, get the full list of mapped business_ids for each user
        pl.col('business_id').map_elements(
            lambda items_list: [business_id_map.get(item) for item in items_list if business_id_map.get(item) is not None],
            return_dtype=pl.List(pl.Int64)
        ).alias('full_mapped_history')
    )
    # DataFrame now has: user_id (string), full_mapped_history (List[Int64])
    .with_columns([
        # Split 'full_mapped_history' into 80% history and 20% prediction
        pl.col('full_mapped_history').map_elements(
            lambda history: history[:math.floor(len(history) * 0.8)], # First 80%
            return_dtype=pl.List(pl.Int64)
        ).alias('given_user_history'),
        pl.col('full_mapped_history').map_elements(
            lambda history: history[math.floor(len(history) * 0.8):], # Last 20%
            return_dtype=pl.List(pl.Int64)
        ).alias('predicting_items'),

        # Map the original 'user_id' (string) to its integer representation.
        # Since no .alias() is used, this replaces the content of the 'user_id' column.
        pl.col('user_id').replace_strict(user_id_map)
    ])
    # DataFrame now has: user_id (Int64), full_mapped_history (List[Int64]),
    #                    given_user_history (List[Int64]), predicting_items (List[Int64])
    .select([
        # Select the final desired columns. This also implicitly drops 'full_mapped_history'.
        pl.col('user_id'),           # This is the mapped integer user ID
        pl.col('given_user_history'),
        pl.col('predicting_items')
    ])
)


In [44]:
print(test_df.head())

shape: (5, 3)
┌─────────┬─────────────────────────┬────────────────────────┐
│ user_id ┆ given_user_history      ┆ predicting_items       │
│ ---     ┆ ---                     ┆ ---                    │
│ i64     ┆ list[i64]               ┆ list[i64]              │
╞═════════╪═════════════════════════╪════════════════════════╡
│ 29771   ┆ [17613, 32049, … 11731] ┆ [10335, 10243, … 6233] │
│ 28870   ┆ [5619, 7283, … 36265]   ┆ [31737, 35196, … 5074] │
│ 27667   ┆ [23015, 21794, … 27343] ┆ [6311, 26188, … 6454]  │
│ 30026   ┆ [2083, 33895, … 41790]  ┆ [17100, 2501, … 20971] │
│ 30302   ┆ [34731, 13795, … 37587] ┆ [25086, 1288, … 8442]  │
└─────────┴─────────────────────────┴────────────────────────┘


In [45]:
# Create training sequences with user history and next item to predict
train_sequences = []
for user_id, group in train_data.sort('date').group_by('user_id'):
    history = [business_id_map[item] for item in group['business_id']]
    user_id = user_id_map[user_id[0]]
    
    # Create sequences of increasing length
    for i in range(1, len(history)):
        train_sequences.append({
            'user_id': user_id,
            'given_user_history': history[:i],
            'predicting_items': history[i]
        })

train_df = pl.DataFrame(train_sequences)
display(train_df.head())

user_id,given_user_history,predicting_items
i64,list[i64],i64
24243,[22650],7267
24243,"[22650, 7267]",13905
24243,"[22650, 7267, 13905]",9658
24243,"[22650, 7267, … 9658]",43155
24243,"[22650, 7267, … 43155]",3627


In [46]:
# check the data types of the columns
print(train_df.dtypes)

[Int64, List(Int64), Int64]


In [47]:
train_df_to_save = train_df.with_columns(
    pl.col('given_user_history').map_elements(lambda x: str(x.to_list()), return_dtype=pl.String)
)

validation_df_to_save = validation_df.with_columns(
    pl.col('given_user_history').map_elements(lambda x: str(x.to_list()), return_dtype=pl.String),
    pl.col('predicting_items').map_elements(lambda x: str(x.to_list()), return_dtype=pl.String)
)

test_df_to_save = test_df.with_columns(
    pl.col('given_user_history').map_elements(lambda x: str(x.to_list()), return_dtype=pl.String),
    pl.col('predicting_items').map_elements(lambda x: str(x.to_list()), return_dtype=pl.String)
)


# ----- Now, save the modified DataFrames -----
#

train_df_to_save.write_csv(
    os.path.join(target_path, 'training.tsv'), 
    separator='\t'
)
validation_df_to_save.write_csv(
    os.path.join(target_path, 'validation.tsv'), 
    separator='\t'
)
test_df_to_save.write_csv(
    os.path.join(target_path, 'test.tsv'), 
    separator='\t'
)

print("Files saved successfully after converting lists to strings.")

Files saved successfully after converting lists to strings.


In [48]:
# load the saved datasets to verify
train_df_loaded = pd.read_csv('dataset2018/training.tsv', sep='\t')
validation_df_loaded = pd.read_csv('dataset2018/validation.tsv', sep='\t')
test_df_loaded = pd.read_csv('dataset2018/test.tsv', sep='\t')
print("Training Data Loaded:")
print(train_df_loaded.head())
print("Validation Data Loaded:")
print(validation_df_loaded.head())
print("Test Data Loaded:")
print(test_df_loaded.head())

Training Data Loaded:
   user_id                 given_user_history  predicting_items
0    24243                            [22650]              7267
1    24243                      [22650, 7267]             13905
2    24243               [22650, 7267, 13905]              9658
3    24243         [22650, 7267, 13905, 9658]             43155
4    24243  [22650, 7267, 13905, 9658, 43155]              3627
Validation Data Loaded:
   user_id                                 given_user_history   
0    27178  [1184, 21869, 30940, 5692, 2803, 25888, 30961,...  \
1    24515  [13969, 25826, 24704, 12290, 13158, 7120, 2324...   
2    26371  [21279, 31515, 17892, 35806, 9643, 7530, 24608...   
3    27393  [24749, 34941, 15167, 1933, 34907, 30875, 3657...   
4    27254  [36373, 21766, 15068, 41116, 26764, 35823, 193...   

                                    predicting_items  
0  [19064, 8977, 36844, 24293, 7050, 15629, 7337,...  
1                 [2290, 25693, 40086, 14255, 18887]  
2  [24439, 323

In [49]:
# print the data type of the columns    
print("Data types in loaded training DataFrame:")
print(train_df_loaded.dtypes)
print("Data types in loaded validation DataFrame:")
print(validation_df_loaded.dtypes)
print("Data types in loaded test DataFrame:")
print(test_df_loaded.dtypes)


Data types in loaded training DataFrame:
user_id                int64
given_user_history    object
predicting_items       int64
dtype: object
Data types in loaded validation DataFrame:
user_id                int64
given_user_history    object
predicting_items      object
dtype: object
Data types in loaded test DataFrame:
user_id                int64
given_user_history    object
predicting_items      object
dtype: object
